# Workflow 4: Population-Level Detection of Misfolding Involving Native Entanglements

## Goal

Use proteome-scale structures and LiP-MS-derived labels to test enrichment of conformational signals in native entangled regions and identify high-risk protein subpopulations.

This notebook follows the schema of Workflows 1-3 and mirrors [workflow4_population.md](workflow4_population.md).

### Runtime notes
- Native NCLE batch over the full tutorial cohort can take hours to days.
- This notebook is configured to run the full tutorial cohort (no top-20 subselect).
- Regression and Monte Carlo run directly from the full-cohort outputs.

> Environment: activate `ncledetector` and select that kernel before running cells.

In [ ]:
import os,sys
import subprocess
from pathlib import Path
import pandas as pd

DATASTORE = "/home/ians/STORE/NCLEdetector"
SCRIPTSDIR = "/home/ians/git_repos/NCLEdetector"
STRUCTDIR = f"{DATASTORE}/user_input/proteome_structures"
OUTDIR = f"{DATASTORE}/outputs/notebook_tmp/workflow4"
GENELIST_AF = f"{DATASTORE}/user_input/experimental_data/Gene_lists/AF/AF_0.6g_C_Rall_spa50_LiPMScov50_ent_genes.txt"
RES_FEATURES_AF = f"{DATASTORE}/user_input/experimental_data/PDB_residue_features/AF/residueFeatures.csv"

NATIVE_BATCH_OUTDIR = f"{DATASTORE}/outputs/workflow4/nativeNCLE_all"
DESIGN_MATRIX_FILE = f"{SCRIPTSDIR}/assets/residue_dataframes_workflow4.csv"
REG_OUTDIR = f"{OUTDIR}/population_modeling"
MC_OUTDIR = f"{OUTDIR}/monte_carlo"
LOGDIR = f"{OUTDIR}/logs"

for p in [NATIVE_BATCH_OUTDIR, REG_OUTDIR, MC_OUTDIR, LOGDIR]:
    os.makedirs(p, exist_ok=True)

print("DATASTORE:", DATASTORE)
print("STRUCTDIR:", STRUCTDIR)
print("OUTDIR:", OUTDIR)
print("GENELIST_AF:", GENELIST_AF)
print("RES_FEATURES_AF:", RES_FEATURES_AF)
print("\nOutput directories ready.")

## Step 0. Activate your environment and set paths

Activate the `ncledetector` conda environment before running this notebook:

```bash
source ~/.bashrc
conda activate ncledetector
```

The next cells define all tutorial paths, create the output directories, and verify that the required AlphaFold structures and gene-list inputs are present.

### Required input files

| File / directory | Role |
|------------------|------|
| `$DATASTORE/user_input/proteome_structures/AF/` | AlphaFold structure library |
| `$DATASTORE/user_input/proteome_structures/EXP/` | Experimental structure library |
| `$DATASTORE/user_input/experimental_data/Gene_lists/AF/...txt` | Tutorial cohort gene list |
| `$DATASTORE/user_input/experimental_data/PDB_residue_features/AF/residueFeatures.csv` | Per-residue feature table (`cut_C_Rall`, `AA`, etc.) |
| `assets/residue_dataframes_workflow4.csv` | Combined residue design matrix (built in Step 1) |

### Tutorial cohort selection

Use the full tutorial AF ent-gene list for batch NCLE and downstream modeling.

In [ ]:
genes_full = genes_all
print(f"Using full tutorial gene list with {len(genes_full)} genes")
print("Gene sample:", genes_full[:5])

## save first 5 genes to a /tmp file for the next cell
GENELIST_AF_SHORT = "/tmp/genes_sample.txt"
with open(GENELIST_AF_SHORT, "w") as f:
    for g in genes_full[:5]:
        f.write(g + "\n")
print(f"Saved first 5 genes to {GENELIST_AF_SHORT}")

## Step 1. Run Workflow 1 in batch to generate native NCLEs per protein

Run the single-script wrapper `run_workflow4_nativeNCLE_batch.py` to:

1. Execute native NCLE analysis in parallel for each protein in the gene list (calls `scripts/run_nativeNCLE.py` per protein).
2. Build the combined residue-level design matrix from per-residue feature inputs plus the NCLE-derived `region` labels.

**Selection rule:** A PDB file is processed only if its stem (filename without `.pdb`) is present in `--gene_list`.

Per-protein outputs are written under `$OUTDIR/nativeNCLE_all/<ID>/` with four subdirectories:

```
<ID>/
├── Native_GE/
├── Native_HQ_GE/
├── Native_clustered_HQ_GE/
└── Native_clustered_HQ_GE_features/
```

The combined design matrix (`residue_dataframes_workflow4.csv`) is generated automatically after all per-protein jobs finish and is the direct input for Steps 2 and 3.

> **Runtime note:** Batch NCLE generation over a full proteome can take hours to days. This notebook runs on a small tutorial cohort (first 5 genes) for speed.

In [ ]:
import shlex
import subprocess

run_cmd = [
    "python", "scripts/run_workflow4_nativeNCLE_batch.py",
    "--pdb_dir", f"{STRUCTDIR}/AF",
    "--gene_list", GENELIST_AF_SHORT,
    "--outdir", f"{OUTDIR}/nativeNCLE_all",
    "--organism", "Ecoli",
    "--model", "AF",
    "--ent_detection_method", "3",
    "--g_threshold", "0.6",
    "--density", "1.0",
    "--residue_features_file", RES_FEATURES_AF,
    "--reg_formula", "cut_C_Rall ~ AA + region",
    "--design_matrix_file", f"/tmp/residue_dataframes_workflow4.csv",
    "--nproc", "8",
    "--logdir", LOGDIR,
]

print("Running batch nativeNCLE for full tutorial cohort...")
print(shlex.join(run_cmd))

proc = subprocess.run(
    run_cmd,
    cwd=SCRIPTSDIR,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    check=False,
 )
print(proc.stdout[-4000:])
print("Return code:", proc.returncode)


In [ ]:
out_root = Path(NATIVE_BATCH_OUTDIR)
protein_dirs = sorted([p for p in out_root.iterdir() if p.is_dir()])
print(f"Per-protein output dirs found: {len(protein_dirs)}")
print([p.name for p in protein_dirs[:10]])

# Quick output integrity check
required_subdirs = [
    "Native_GE",
    "Native_HQ_GE",
    "Native_clustered_HQ_GE",
    "Native_clustered_HQ_GE_features",
]

if protein_dirs:
    sample = protein_dirs[0]
    print(f"\nSample protein dir: {sample}")
    for sd in required_subdirs:
        print(sd, (sample / sd).exists())

## Step 2. Test proteome-level enrichment of experimental signal in native entangled regions

Use `ProteomeLogisticRegression` to test whether residues inside native entangled regions (`region = 1`) are enriched for the experimental conformational-change signal (`cut_C_Rall`) while controlling for amino-acid composition.

The regression formula `cut_C_Rall ~ AA + region` fits a logistic model where:
- `region` is the primary test variable (1 = inside entangled region, 0 = outside)
- `AA` controls for amino-acid identity biases

**Interpreting the `region` coefficient:**

$$\text{OR}_{region} = e^{\beta_{region}}$$

| OR | p-value | Interpretation |
|----|---------|----------------|
| > 1 | significant | Enrichment — entangled residues are over-represented for the experimental signal |
| < 1 | significant | Depletion — entangled residues are under-represented |
| any | non-significant | No detectable association in this model |

**Output:** `$OUTDIR/population_modeling/regression_results_Ecoli_population.csv`

In [ ]:
reg_cmd = [
    "python", "scripts/run_population_modeling.py",
    "--dataframe_files", DESIGN_MATRIX_FILE,
    "--outdir", REG_OUTDIR,
    "--gene_list", GENELIST_AF,
    "--ID", "Ecoli_population",
    "--reg_formula", "cut_C_Rall ~ AA + region",
    "--logdir", LOGDIR,
]

print("Running proteome regression...")
print(shlex.join(reg_cmd))

proc = subprocess.run(
    reg_cmd,
    cwd=SCRIPTSDIR,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    check=False,
 )
print(proc.stdout[-4000:])
print("Return code:", proc.returncode)


## Step 3. Monte Carlo subpopulation selection (full tutorial cohort)

Use `MonteCarlo` to partition the protein cohort into `n_groups` subpopulations and optimize an objective combining enrichment signal and group-size distribution constraints.

The objective minimizes:

$$E = -C_1 \cdot \log(\text{OR}) + C_2 \cdot \text{KS}$$

where OR is the odds ratio for `region` in the current protein group and KS is the Kolmogorov-Smirnov statistic penalizing deviation of group-size distribution from the target.

| Parameter | Meaning |
|-----------|---------|
| `n_groups` | Number of protein subpopulations to infer |
| `steps` | Total Monte Carlo updates (100,000 recommended for production) |
| `C1` | Weight on the enrichment-signal component |
| `C2` | Weight on the size-distribution penalty component |
| `beta` | Inverse-temperature controlling acceptance probability |

> **Runtime:** A single production run (100,000 steps) takes hours. This tutorial uses `steps=10000` for speed — steady states are just starting to form at this scale.

In [ ]:
mc_cmd = [
    "python", "scripts/run_montecarlo.py",
    "--dataframe_files", DESIGN_MATRIX_FILE,
    "--outdir", MC_OUTDIR,
    "--gene_list", GENELIST_AF,
    "--ID", "Ecoli_population_mc",
    "--reg_formula", "cut_C_Rall ~ region + AA",
    "--response_var", "cut_C_Rall",
    "--test_var", "region",
    "--n_groups", "4",
    "--steps", "10000",
    "--C1", "1.0",
    "--C2", "2.5",
    "--beta", "0.05",
    "--logdir", LOGDIR,
]

print("Running Monte Carlo...")
print(shlex.join(mc_cmd))

proc = subprocess.run(
    mc_cmd,
    cwd=SCRIPTSDIR,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    check=False,
 )
print(proc.stdout[-4000:])
print("Return code:", proc.returncode)


In [ ]:
mc_out = Path(MC_OUTDIR)
final_files = sorted(mc_out.glob("Final_step_reg_*"))
state_gene_files = sorted(mc_out.glob("State*_final_genelist_*"))
state_traj_files = sorted(mc_out.glob("State*_final_traj_*"))

print("Monte Carlo output summary:")
print("  Final reg files:", len(final_files))
print("  Final genelist files:", len(state_gene_files))
print("  Final trajectory files:", len(state_traj_files))

print("\nSample outputs:")
for p in (final_files[:2] + state_gene_files[:2] + state_traj_files[:2]):
    print(" ", p.name)

## Step 4. Rank and refine candidate protein groups

Inspect the trajectory files for each state and look for stabilization of the odds ratio and energies. If all states have reached a steady state you can stop the protocol. In this tutorial we only ran 10,000 MC steps — steady states are starting to form. Recommend 100,000 steps for production runs.

One figure per state shows how each optimization parameter evolves over MC steps. Use these plots to tune `C1`, `C2`, and `beta` for your system before production runs.

### Building robust candidate sets from repeated runs

1. Run Monte Carlo multiple times with different random seeds or independent starts.
2. For each run, identify the top-enrichment state by final odds ratio.
3. Count per-protein inclusion frequency in top states across runs.
4. Keep proteins with high reproducibility (e.g., >70% top-state inclusion).

This produces a stable, high-confidence subpopulation for downstream biological interpretation.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import pandas as pd
from pathlib import Path

mc_out = Path(MC_OUTDIR)
traj_files = sorted(mc_out.glob("State*_final_traj_*.csv"))

if not traj_files:
    print("No trajectory files found. Run Step 4 first.")
else:
    for traj_file in traj_files:
        df = pd.read_csv(traj_file)
        state = int(df["state"].iloc[0])
        steps = df["step"].values

        fig = plt.figure(figsize=(15, 8))
        fig.suptitle(f"State {state} — MC parameter evolution\n{traj_file.name}", fontsize=13, fontweight="bold")
        gs = gridspec.GridSpec(2, 3, figure=fig, hspace=0.45, wspace=0.38)

        # ── row 0 ──────────────────────────────────────────────────────────────
        ax0 = fig.add_subplot(gs[0, 0])
        ax0.plot(steps, df["OR"], color="steelblue", lw=1.2)
        ax0.axhline(1.0, color="grey", ls="--", lw=0.8)
        ax0.set_xlabel("MC step")
        ax0.set_ylabel("Odds ratio (OR)")
        ax0.set_title("Odds ratio")

        ax1 = fig.add_subplot(gs[0, 1])
        ax1.plot(steps, df["pvalue"], color="darkorange", lw=1.2)
        ax1.axhline(0.05, color="grey", ls="--", lw=0.8, label="p=0.05")
        ax1.set_xlabel("MC step")
        ax1.set_ylabel("p-value")
        ax1.set_title("Fisher's exact p-value")
        ax1.legend(fontsize=8)

        ax2 = fig.add_subplot(gs[0, 2])
        ax2.plot(steps, df["E"], color="firebrick", lw=1.2)
        ax2.set_xlabel("MC step")
        ax2.set_ylabel("Energy  E")
        ax2.set_title("Energy  (E = −C₁·log OR + C₂·KS)")

        # ── row 1 ──────────────────────────────────────────────────────────────
        ax3 = fig.add_subplot(gs[1, 0])
        ax3.plot(steps, df["ks_stat_size"], color="purple", lw=1.2)
        ax3.set_xlabel("MC step")
        ax3.set_ylabel("KS statistic")
        ax3.set_title("Size-distribution KS stat")

        ax4 = fig.add_subplot(gs[1, 1])
        ax4.plot(steps, df["psize_mean"], color="seagreen", lw=1.2, label="mean")
        ax4.fill_between(steps, df["psize_lb"], df["psize_ub"],
                         color="seagreen", alpha=0.25, label="95 % CI")
        ax4.set_xlabel("MC step")
        ax4.set_ylabel("Protein length (aa)")
        ax4.set_title("Mean protein size")
        ax4.legend(fontsize=8)

        ax5 = fig.add_subplot(gs[1, 2])
        ax5.plot(steps, df["beta"], color="saddlebrown", lw=1.2)
        ax5.set_xlabel("MC step")
        ax5.set_ylabel("β (inverse temperature)")
        ax5.set_title("Annealing schedule  β")

        plt.show()
        print()


## Step 5. Next actions

- Aggregate repeated Monte Carlo runs and rank proteins by inclusion frequency in top-enrichment states.
- Compare top-state protein sets against independent biological annotations.
- Tune `n_groups`, `C1`, `C2`, and `beta` based on your desired tradeoff between enrichment and size-distribution matching.